In [25]:
import asyncio
import time
import os
import json
import hashlib
from typing import List, Dict, Tuple
from urllib.parse import urlparse
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from crawl4ai.deep_crawling import BFSDeepCrawlStrategy
from bs4 import BeautifulSoup
from firecrawl import FirecrawlApp
from google.oauth2 import service_account
from googleapiclient.discovery import build


CATEGORY_THRESHOLDS = {
    "ABOUT_US": 200,
    "EBOOK": 200,
    "COURSES": 300,
    "RECENT_BLOG": 450,
    "TESTIMONIALS": 100,
    "WEBINAR": 150,
    "SERVICES": 150,
    "PODCAST": 200,
    "SHOP": 100,
}

CATEGORY_KEYWORDS = {
    "ABOUT_US": ["about", "who-we-are", "company", "our-story"],
    "EBOOK": ["ebook", "e-book", "downloads", "whitepaper"],
    "COURSES": ["course", "academy", "learning"],
    "RECENT_BLOG": ["blog", "insights", "articles"],
    "TESTIMONIALS": ["testimonial", "reviews", "case-study"],
    "WEBINAR": ["webinar", "event", "session"],
    "SERVICES": ["service", "solution", "capability"],
    "PODCAST": ["podcast", "listen", "episodes"],
    "SHOP": ["shop", "store", "buy"]
}

COLUMN_TO_READ_URL_FROM = "G"
COLUMN_TO_WRITE_URL_TO = {
    "ABOUT_US": "M",
    "EBOOK": "N",
    "COURSES": "O",
    "RECENT_BLOG": "P",
    "TESTIMONIALS": "Q",
    "WEBINAR": "R",
    "SERVICES": "S",
    "PODCAST": "T",
    "SHOP": "U"
}

CACHE_DIR = "firecrawl_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CATEGORY_RULES = {
    'ABOUT_US': 'ascending',
    'EBOOK': 'ascending',
    'COURSES': 'ascending',
    'RECENT_BLOG': 'descending',
    'TESTIMONIALS': 'ascending',
    'WEBINAR': 'descending',
    'SERVICES': 'descending',
    'PODCAST': 'descending',
    'SHOP': 'ascending'
}

EXTRACTION_METADATA_COLUMN = "V"  # Column V for metadata

# Calculate URL Depth
def calculate_url_depth(url: str) -> int:
    try:
        parsed = urlparse(url)
        path = parsed.path.strip('/').split('/')
        return len(path)
    except Exception:
        return -1  # Invalid URL, will be skipped

# Deepest Point Function
def deepest_point_function(url_depth_pairs: List[Tuple[str, int]], category: str) -> List[str]:
    if category not in CATEGORY_RULES:
        raise ValueError(f"Category {category} not found in CATEGORY_RULES")
    
    sort_order = CATEGORY_RULES[category]
    if sort_order == 'ascending':
        top_urls = sorted(url_depth_pairs, key=lambda x: x[1])[:10]
    else:  # descending
        top_urls = sorted(url_depth_pairs, key=lambda x: x[1], reverse=True)[:10]
    
    return [url for url, _ in top_urls]

# FirecrawlWrapper
class FirecrawlWrapper:
    def __init__(self, api_key):
        self.app = FirecrawlApp(api_key=api_key)

    def _hash_url(self, url: str) -> str:
        return hashlib.md5(url.encode()).hexdigest()

    def _get_cache_path(self, url: str) -> str:
        return os.path.join(CACHE_DIR, f"{self._hash_url(url)}.json")

    def map_url(self, url: str) -> List[str]:
        cache_path = self._get_cache_path(url)
        if os.path.exists(cache_path):
            with open(cache_path, 'r') as f:
                links = json.load(f)
                print(f"Loaded {len(links)} cached links for {url}")
                return links
        try:
            result = self.app.map_url(url)
            if getattr(result, 'success', False):
                links = result.links
                with open(cache_path, 'w') as f:
                    json.dump(links, f, indent=2)
                print(f"Firecrawl found {len(links)} links for {url}")
                return links
            else:
                print(f"Firecrawl failed for {url}")
                return []
        except Exception as e:
            print(f"Firecrawl error for {url}: {e}")
            return []

    def filter_by_category(self, urls: List[str], category: str) -> List[str]:
        keywords = CATEGORY_KEYWORDS.get(category.upper(), [])
        if not keywords:
            print(f"No keywords defined for category {category}")
            return []
        return [u for u in urls if any(k in u.lower() for k in keywords)]

# extract_main_html_content
def extract_main_html_content(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    main = soup.find("main") or soup.find("article")
    if main:
        return main.get_text(separator="\n", strip=True)
    candidates = [
        div for div in soup.find_all("div")
        if len(div.get_text(strip=True)) > 200
           and not any(c in " ".join(div.get("class", [])).lower() for c in ["nav", "header", "footer", "popup"])
    ]
    if candidates:
        return max(candidates, key=lambda d: len(d.get_text(strip=True))).get_text(separator="\n", strip=True)
    return soup.get_text(separator="\n", strip=True)

# crawl_and_select_content
async def crawl_and_select_content(urls: List[str]) -> List[str]:
    crawler_config = CrawlerRunConfig(
        deep_crawl_strategy=BFSDeepCrawlStrategy(max_depth=0),
        verbose=False
    )
    contents = []
    async with AsyncWebCrawler() as crawler:
        for url in urls:
            try:
                result = await asyncio.wait_for(crawler.arun(url, config=crawler_config), timeout=15)
                if result and result[0].html:
                    text = extract_main_html_content(result[0].html)
                    contents.append(text)
            except Exception as e:
                print(f"Error crawling {url}: {e}")
    return contents

# GoogleSheetsManager
class GoogleSheetsManager:
    def __init__(self, credentials_file: str):
        scopes = ['https://www.googleapis.com/auth/spreadsheets']
        creds = service_account.Credentials.from_service_account_file(credentials_file, scopes=scopes)
        self.service = build('sheets', 'v4', credentials=creds)

    def extract_spreadsheet_id(self, sheet_url: str) -> str:
        import re
        pattern = r'/spreadsheets/d/([a-zA-Z0-9-_]+)'
        match = re.search(pattern, sheet_url)
        if match:
            return match.group(1)
        raise ValueError("Invalid Google Sheet URL")

    def get_urls(self, spreadsheet_id: str) -> List[Dict]:
        range_name = f"{COLUMN_TO_READ_URL_FROM}2:{COLUMN_TO_READ_URL_FROM}"
        result = self.service.spreadsheets().values().get(spreadsheetId=spreadsheet_id, range=range_name).execute()
        values = result.get('values', [])
        return [(i + 2, row[0]) for i, row in enumerate(values) if row and row[0].strip()]

    def update_result(self, spreadsheet_id: str, row: int, content: str, metadata: str, column_to_process: str):
        # Write content to COLUMN_TO_PROCESS
        content_range = f"{column_to_process}{row}"
        if len(content) > 50000:
            print(f"Truncating content from {len(content)} to 50000 characters.")
            content = content[:50000]
        self.service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=content_range,
            valueInputOption='RAW',
            body={'values': [[content]]}
        ).execute()
        print(f"Updated row {row} in column {column_to_process} with content")

        # Read existing metadata from EXTRACTION_METADATA_COLUMN
        metadata_range = f"{EXTRACTION_METADATA_COLUMN}{row}"
        try:
            existing_metadata = self.service.spreadsheets().values().get(
                spreadsheetId=spreadsheet_id,
                range=metadata_range
            ).execute().get('values', [['']])[0][0]
        except Exception as e:
            print(f"Error reading existing metadata for row {row}: {e}")
            existing_metadata = ''

        # Parse existing metadata and update or append new metadata
        category = metadata.split('=')[0]  # Extract category from new metadata (e.g., 'PODCAST')
        if existing_metadata:
            metadata_parts = existing_metadata.split(',')
            updated_parts = []
            category_found = False
            for part in metadata_parts:
                if part.startswith(category + '='):
                    # Update existing category with new numerical value
                    updated_parts.append(metadata)
                    category_found = True
                else:
                    updated_parts.append(part)
            if not category_found:
                # Append new metadata if category not found
                updated_parts.append(metadata)
            new_metadata = ','.join(updated_parts)
        else:
            new_metadata = metadata

        # Write updated metadata to EXTRACTION_METADATA_COLUMN
        self.service.spreadsheets().values().update(
            spreadsheetId=spreadsheet_id,
            range=metadata_range,
            valueInputOption='RAW',
            body={'values': [[new_metadata]]}
        ).execute()
        print(f"Updated row {row} in column {EXTRACTION_METADATA_COLUMN} with metadata: {new_metadata}")

# Main Processing Function
async def process_all_rows_firecrawl(sheet_url: str, credentials_file: str, firecrawl_api_key: str, category: str):
    sheet_mgr = GoogleSheetsManager(credentials_file)
    spreadsheet_id = sheet_mgr.extract_spreadsheet_id(sheet_url)
    urls = sheet_mgr.get_urls(spreadsheet_id)

    firecrawl = FirecrawlWrapper(api_key=firecrawl_api_key)

    # Use COLUMN_TO_WRITE_URL_TO directly for COLUMN_TO_PROCESS
    column_to_process = COLUMN_TO_WRITE_URL_TO.get(category.upper())
    if not column_to_process:
        raise ValueError(f"No column defined for category {category}")

    for i, (row_num, main_url) in enumerate(urls):
        print(f"\nProcessing row {row_num}: {main_url}")
        sub_urls = firecrawl.map_url(main_url)
        await asyncio.sleep(6.5)

        filtered = firecrawl.filter_by_category(sub_urls, category)

        if not filtered:
            # Write "No URL found" to COLUMN_TO_PROCESS and "CATEGORY=0" to EXTRACTION_METADATA_COLUMN
            sheet_mgr.update_result(spreadsheet_id, row_num, "No URL found", f"{category.upper()}=0", column_to_process)
            continue

        # Calculate depths and create (url, depth) pairs
        url_depth_pairs = []
        for url in filtered:
            depth = calculate_url_depth(url)
            if depth != -1:
                url_depth_pairs.append((url, depth))

        if not url_depth_pairs:
            # No valid URLs after depth calculation
            sheet_mgr.update_result(spreadsheet_id, row_num, "No URL found", f"{category.upper()}=0", column_to_process)
            continue

        # Sort based on CATEGORY_RULES
        sort_order = CATEGORY_RULES.get(category.upper())
        if not sort_order:
            raise ValueError(f"No sorting rule defined for category {category}")

        reverse_sort = sort_order == 'descending'
        sorted_url_depth_pairs = sorted(url_depth_pairs, key=lambda x: x[1], reverse=reverse_sort)

        num_urls = len(sorted_url_depth_pairs)
        if num_urls <= 10:
            # Take all URLs
            selected_urls = [url for url, _ in sorted_url_depth_pairs]
        else:
            # Select top 10 based on deepest_point_function
            selected_urls = deepest_point_function(sorted_url_depth_pairs, category.upper())

        # Crawl selected URLs and collect content
        contents = await crawl_and_select_content(selected_urls)

        if not contents:
            content = "No meaningful content found"
            metadata = f"{category.upper()}=0"
        else:
            content = " --- NEXT CONTENT FROM HERE --- ".join(contents)
            metadata = f"{category.upper()}={len(contents)}"

        # Write to Google Sheet
        sheet_mgr.update_result(spreadsheet_id, row_num, content, metadata, column_to_process)

In [26]:
FIRECRAWL_API="fc-29599096ac8b426dbf178180c53500ed"
CREDENTIALS_FILE = 'url-to-email-445616-cebe4868914f.json'
GOOGLE_SHEET_URL = "https://docs.google.com/spreadsheets/d/1QtKOB5ChRemg2_wJOxeZhn1qao1ZrRjVK8nExVzbxEI/edit?gid=2011509251#gid=2011509251" 

In [27]:
await process_all_rows_firecrawl(
    sheet_url=GOOGLE_SHEET_URL,
    credentials_file=CREDENTIALS_FILE,
    firecrawl_api_key=FIRECRAWL_API,
    category="PODCAST"
)


Processing row 3: https://redkiteproject.com
Loaded 43 cached links for https://redkiteproject.com
Updated row 3 in column T with content
Updated row 3 in column V with metadata: PODCAST=0

Processing row 4: https://lrscpa.com
Loaded 429 cached links for https://lrscpa.com


[INIT].... → Crawl4AI 0.6.3 

Updated row 4 in column T with content
Updated row 4 in column V with metadata: PODCAST=2

Processing row 5: https://ashtae.com
Loaded 262 cached links for https://ashtae.com
Updated row 5 in column T with content
Updated row 5 in column V with metadata: PODCAST=0

Processing row 6: https://aithonsolutions.com
Loaded 27 cached links for https://aithonsolutions.com
Updated row 6 in column T with content
Updated row 6 in column V with metadata: PODCAST=0

Processing row 7: https://hi-link.com
Loaded 102 cached links for https://hi-link.com
Updated row 7 in column T with content
Updated row 7 in column V with metadata: PODCAST=0

Processing row 8: https://edaptschools.com
Loaded 18 cached links for https://edaptschools.com
Updated row 8 in column T with content
Updated row 8 in column V with metadata: PODCAST=0

Processing row 9: https://globalmetalfinishing.com
Loaded 101 cached links for https://globalmetalfinishing.com
Updated row 9 in column T with content
Updated row 9 in column V wi

[INIT].... → Crawl4AI 0.6.3 

Truncating content from 57766 to 50000 characters.
Updated row 10 in column T with content
Updated row 10 in column V with metadata: PODCAST=10

Processing row 11: https://horizon-five.com
Firecrawl found 16 links for https://horizon-five.com
Updated row 11 in column T with content
Updated row 11 in column V with metadata: PODCAST=0

Processing row 12: https://acfamilyoffice.com
Firecrawl found 70 links for https://acfamilyoffice.com
Updated row 12 in column T with content
Updated row 12 in column V with metadata: PODCAST=0

Processing row 13: https://greenseedtech.com
Firecrawl found 4 links for https://greenseedtech.com
Updated row 13 in column T with content
Updated row 13 in column V with metadata: PODCAST=0

Processing row 14: https://slabstack.com
Firecrawl found 37 links for https://slabstack.com
Updated row 14 in column T with content
Updated row 14 in column V with metadata: PODCAST=0

Processing row 15: https://evolvcompass.com
Firecrawl found 77 links for https://evolvcompas

CancelledError: 